In [15]:
import os
from google.colab import drive

mountpoint = '/content/drive'


drive.mount(mountpoint)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from google.colab import drive

!pip install -q transformers datasets accelerate evaluate seqeval
!pip install -q "torch>=1.13"
!pip install -q imbalanced-learn

import os, json, numpy as np, pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

PROJECT_ROOT = "/content/drive/My Drive/radiological_report"
DATA_PROC_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
HF_SAVE_DIR = os.path.join(PROJECT_ROOT, "models", "tl")
os.makedirs(HF_SAVE_DIR, exist_ok=True)
print("Paths set. HF_SAVE_DIR:", HF_SAVE_DIR)


Paths set. HF_SAVE_DIR: /content/drive/My Drive/radiological_report/models/tl


In [17]:
import os

PROJECT_ROOT = "/content/drive/My Drive/radiological_report"
DATA_PROC_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

os.makedirs(DATA_PROC_DIR, exist_ok=True)
print(f"Created: {DATA_PROC_DIR}")


Created: /content/drive/My Drive/radiological_report/data/processed


In [18]:
#  C3

train_csv_path = os.path.join(DATA_PROC_DIR, "train.csv")
val_csv_path = os.path.join(DATA_PROC_DIR, "val.csv")
test_csv_path = os.path.join(DATA_PROC_DIR, "test.csv")

if not os.path.exists(DATA_PROC_DIR):
    print(f"Error: The directory '{DATA_PROC_DIR}' does not exist in your Google Drive.")
    print("Please make sure you have created this directory and placed the necessary CSV files inside it.")
    raise FileNotFoundError(f"Directory not found: {DATA_PROC_DIR}")

required_files = ["train.csv", "val.csv", "test.csv"]
missing_files = []
for filename in required_files:
    file_path = os.path.join(DATA_PROC_DIR, filename)
    if not os.path.exists(file_path):
        missing_files.append(filename)

if missing_files:
    print(f"Error: The following files are missing from '{DATA_PROC_DIR}': {', '.join(missing_files)}.")
    print("Please upload these files to your Google Drive at the specified path.")
    print(f"Current contents of '{DATA_PROC_DIR}': {os.listdir(DATA_PROC_DIR)}")
    raise FileNotFoundError(f"Missing required CSV files: {', '.join(missing_files)}")

train_df = pd.read_csv(train_csv_path)[['report_clean','label']].rename(columns={'report_clean':'text'})
val_df = pd.read_csv(val_csv_path)[['report_clean','label']].rename(columns={'report_clean':'text'})
test_df = pd.read_csv(test_csv_path)[['report_clean','label']].rename(columns={'report_clean':'text'})

hf_train = Dataset.from_pandas(train_df)
hf_val = Dataset.from_pandas(val_df)
hf_test = Dataset.from_pandas(test_df)
hf_ds = DatasetDict({"train": hf_train, "validation": hf_val, "test": hf_test})

for split in hf_ds:
    if 'label' in hf_ds[split].features and hf_ds[split].features['label'].dtype != 'int64':
        print(f"Casting 'label' column in {split} split to int64 from {hf_ds[split].features['label'].dtype}.")
        hf_ds[split] = hf_ds[split].map(lambda example: {'label': int(example['label'])}, batched=False)

print(hf_ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2335
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 501
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 501
    })
})


In [19]:
import pandas as pd

train_df_clean = train_df.drop_duplicates(subset=['text']).reset_index(drop=True)
val_df_clean = val_df.drop_duplicates(subset=['text']).reset_index(drop=True)
test_df_clean = test_df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"After deduplication:")
print(f"Train: {len(train_df)} -> {len(train_df_clean)}")
print(f"Val: {len(val_df)} -> {len(val_df_clean)}")
print(f"Test: {len(test_df)} -> {len(test_df_clean)}")


train_texts = set(train_df_clean['text'].values)
val_texts = set(val_df_clean['text'].values)
test_texts = set(test_df_clean['text'].values)

print(f"\nCross-set overlaps:")
print(f"Train-Val overlap: {len(train_texts & val_texts)}")
print(f"Train-Test overlap: {len(train_texts & test_texts)}")
print(f"Val-Test overlap: {len(val_texts & test_texts)}")

val_df_clean = val_df_clean[~val_df_clean['text'].isin(train_texts)].reset_index(drop=True)
test_df_clean = test_df_clean[~test_df_clean['text'].isin(train_texts)].reset_index(drop=True)
test_df_clean = test_df_clean[~test_df_clean['text'].isin(val_texts)].reset_index(drop=True)

print(f"\nAfter removing cross-set leakage:")
print(f"Train: {len(train_df_clean)}")
print(f"Val: {len(val_df_clean)}")
print(f"Test: {len(test_df_clean)}")


hf_train = Dataset.from_pandas(train_df_clean)
hf_val = Dataset.from_pandas(val_df_clean)
hf_test = Dataset.from_pandas(test_df_clean)
hf_ds = DatasetDict({"train": hf_train, "validation": hf_val, "test": hf_test})
print(hf_ds)

After deduplication:
Train: 2335 -> 1831
Val: 501 -> 431
Test: 501 -> 425

Cross-set overlaps:
Train-Val overlap: 63
Train-Test overlap: 69
Val-Test overlap: 39

After removing cross-set leakage:
Train: 1831
Val: 368
Test: 353
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1831
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 368
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 353
    })
})


In [21]:

from imblearn.over_sampling import RandomOverSampler

train_df_reduced = train_df_clean.sample(frac=0.6, random_state=42).reset_index(drop=True)

X_train = train_df_reduced.drop('label', axis=1)
y_train = train_df_reduced['label']

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

train_df_resampled = pd.DataFrame(X_resampled, columns=X_train.columns)
train_df_resampled['label'] = y_resampled

hf_train = Dataset.from_pandas(train_df_resampled)
hf_val = Dataset.from_pandas(val_df_clean)
hf_test = Dataset.from_pandas(test_df_clean)
hf_ds = DatasetDict({"train": hf_train, "validation": hf_val, "test": hf_test})

print(f"Updated dataset sizes (after oversampling minority class in training data):")
print(f"Train: {len(train_df_resampled)} (from {len(train_df_reduced)} before oversampling)")
print(f"Val: {len(val_df_clean)}")
print(f"Test: {len(test_df_clean)}")
print(hf_ds)


from transformers import AutoConfig

def fine_tune_transformer(model_name, run_name, num_train_epochs=1, per_device_train_batch_size=8, dropout=0.1, weight_decay=0.01):
    print("Fine-tuning", model_name, "->", run_name)

    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    config.hidden_dropout_prob = dropout
    config.attention_probs_dropout_prob = dropout

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    def preprocess(examples):
        return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)
    tokenized = hf_ds.map(preprocess, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    output_dir = os.path.join(HF_SAVE_DIR, run_name)
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        learning_rate=1e-5,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=16,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        gradient_accumulation_steps=1,
        fp16=True,
        logging_steps=50,
        report_to="none"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
        acc = accuracy_score(labels, preds)
        return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_res = trainer.evaluate(tokenized['test'])
    print("Evaluation on test:", eval_res)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    with open(os.path.join(output_dir, "eval_results.json"), "w") as f:
        json.dump(eval_res, f, indent=2)
    return eval_res


Updated dataset sizes (after oversampling minority class in training data):
Train: 2076 (from 1099 before oversampling)
Val: 368
Test: 353
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2076
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 368
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 353
    })
})


In [22]:
# tl models:

res = fine_tune_transformer("dmis-lab/biobert-v1.1", "biobert_v1.1", num_train_epochs=3, per_device_train_batch_size=16, dropout=0.4, weight_decay=0.005)
res = fine_tune_transformer("emilyalsentzer/Bio_ClinicalBERT", "clinicalbert", num_train_epochs=3, per_device_train_batch_size=16, dropout=0.2, weight_decay=0.005)
res = fine_tune_transformer("roberta-base", "roberta_base", num_train_epochs=3, per_device_train_batch_size=16, dropout=0.3, weight_decay=0.005)


Fine-tuning dmis-lab/biobert-v1.1 -> biobert_v1.1


Map:   0%|          | 0/2076 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

Map:   0%|          | 0/353 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.202196,0.089113,3,0.960340,0.818182,0.428571,0.562500


Evaluation on test: {'eval_loss': 0.08911272138357162, 'eval_accuracy': 0.9603399433427762, 'eval_precision': 0.8181818181818182, 'eval_recall': 0.42857142857142855, 'eval_f1': 0.5625}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
import pandas as pd
import os
import json

HF_SAVE_DIR = "/content/drive/My Drive/radiological_report/models/tl"

model_run_names = [
    "biobert_v1.1",
    "clinicalbert",
    "roberta_base"
]

all_results = {}

for model_name in model_run_names:
    results_path = os.path.join(HF_SAVE_DIR, model_name, "eval_results.json")
    if os.path.exists(results_path):
        with open(results_path, "r") as f:
            all_results[model_name] = json.load(f)
    else:
        print(f"Warning: Evaluation results not found for {model_name} at {results_path}")

# Convert the results dictionary to a DataFrame for better display
results_df = pd.DataFrame(all_results).T
results_df.index.name = 'Model'

display(results_df)

,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1
Model,,,,,
biobert_v1.1,0.126599,0.949008,0.800000,0.190476,0.307692
clinicalbert,0.109592,0.977337,0.933333,0.666667,0.777778
roberta_base,0.089113,0.960340,0.818182,0.428571,0.562500


In [24]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, pandas as pd


my_drive = "/content/drive/My Drive"
rad_report = None

for item in os.listdir(my_drive):
    if "radiological" in item.lower():
        rad_report = os.path.join(my_drive, item)
        break

if rad_report is None:
    print("⚠️ radiological_report folder not found in My Drive")
    print("Available folders:")
    for item in os.listdir(my_drive):
        if os.path.isdir(os.path.join(my_drive, item)):
            print(f"   ├── {item}")
else:
    print(f"✓ Found: {rad_report}\n")

    TL_MODELS_DIR = os.path.join(rad_report, "models", "tl")
    RESULTS_DIR = os.path.join(rad_report, "results")

    print(f"TL Models: {TL_MODELS_DIR}")
    print(f"Results: {RESULTS_DIR}\n")

    os.makedirs(RESULTS_DIR, exist_ok=True)


    print("Extracting TL results...\n")
    tl_results = []

    for model_name in ["biobert_v1.1", "clinicalbert", "roberta_base"]:
        json_path = os.path.join(TL_MODELS_DIR, model_name, "eval_results.json")

        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                res = json.load(f)

            tl_results.append({
                "Model": model_name,
                "Accuracy": res.get("eval_accuracy", 0),
                "Precision": res.get("eval_precision", 0),
                "Recall": res.get("eval_recall", 0),
                "F1-Score": res.get("eval_f1", 0),
                "Loss": res.get("eval_loss", 0)
            })
            print(f"✓ {model_name}: {res.get('eval_f1', 0):.4f}")
        else:
            print(f"⚠️ {model_name}: NOT found")

    if tl_results:
        tl_df = pd.DataFrame(tl_results)

        print(f"\n{'='*80}")
        print("SAVING RESULTS")
        print(f"{'='*80}\n")

        # Save CSV
        csv_path = os.path.join(RESULTS_DIR, "tl_results.csv")
        tl_df.to_csv(csv_path, index=False)
        print(f"✓ CSV saved: {csv_path}")
        print(f"  Size: {os.path.getsize(csv_path)} bytes")

        # Save Excel
        try:
            excel_path = os.path.join(RESULTS_DIR, "tl_results.xlsx")
            tl_df.to_excel(excel_path, index=False, sheet_name='TL Models')
            print(f"✓ Excel saved: {excel_path}")
            print(f"  Size: {os.path.getsize(excel_path)} bytes")
        except:
            print("⚠️ Excel save skipped (openpyxl not available)")

        print(f"\n{'='*80}")
        print("TL MODELS RESULTS")
        print(f"{'='*80}\n")
        print(tl_df.to_string(index=False))

        print(f"\n{'='*80}")
        print("✅ RESULTS SAVED IN RESULTS FOLDER")
        print(f"{'='*80}\n")

        print("Files in RESULTS folder:")
        for item in sorted(os.listdir(RESULTS_DIR)):
            print(f"   ├── {item}")
    else:
        print("⚠️ No TL results found to save!")

Mounted at /content/drive
✓ Found: /content/drive/My Drive/radiological_report

TL Models: /content/drive/My Drive/radiological_report/models/tl
Results: /content/drive/My Drive/radiological_report/results

Extracting TL results...

✓ biobert_v1.1: 0.3077
✓ clinicalbert: 0.7778
✓ roberta_base: 0.5625

SAVING RESULTS

✓ CSV saved: /content/drive/My Drive/radiological_report/results/tl_results.csv
  Size: 348 bytes
✓ Excel saved: /content/drive/My Drive/radiological_report/results/tl_results.xlsx
  Size: 5212 bytes

TL MODELS RESULTS

       Model  Accuracy  Precision   Recall  F1-Score     Loss
biobert_v1.1  0.949008   0.800000 0.190476  0.307692 0.126599
clinicalbert  0.977337   0.933333 0.666667  0.777778 0.109592
roberta_base  0.960340   0.818182 0.428571  0.562500 0.089113

✅ RESULTS SAVED IN RESULTS FOLDER

Files in RESULTS folder:
   ├── cm_decision_tree.png
   ├── cm_random_forest.png
   ├── cm_svm.png
   ├── dl_results.csv
   ├── ml_results.csv
   ├── tl_results.csv
   ├── tl_re